# Student Segmentation and Clustering

**Objective:** identify student archetypes from study behavior and wellbeing signals, assign meaningful persona labels, and explain cluster membership with transparent rules.

In [ ]:
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "plotly": "plotly",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "sklearn": "scikit-learn",
    "scipy": "scipy",
}


def is_installed(module_name):
    return importlib.util.find_spec(module_name) is not None


def install_package(package_name):
    commands = [
        [sys.executable, "-m", "pip", "install", "-q", package_name],
        [sys.executable, "-m", "pip", "install", "-q", "--user", package_name],
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--break-system-packages",
            "--user",
            package_name,
        ],
    ]
    for cmd in commands:
        try:
            subprocess.check_call(cmd)
            return True
        except subprocess.CalledProcessError:
            continue
    return False


missing = [mod for mod in REQUIRED_PACKAGES if not is_installed(mod)]
installed_any = False
failed = []

if missing:
    print(f"Missing libraries: {missing}")

for module_name in missing:
    package_name = REQUIRED_PACKAGES[module_name]
    print(f"Installing {package_name}...")
    ok = install_package(package_name)
    if ok:
        installed_any = True
    else:
        failed.append(package_name)

if failed:
    raise RuntimeError(
        "Could not install required libraries: "
        + ", ".join(failed)
        + ". Install manually and rerun this cell."
    )

if installed_any:
    print("Dependencies installed. If imports fail, restart kernel and rerun all cells.")
else:
    print("All required dependencies are already installed.")


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import matplotlib.pyplot as plt
import seaborn as sns

from scipy.cluster.hierarchy import linkage, dendrogram

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
    accuracy_score,
    confusion_matrix,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text


pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)
sns.set_theme(style="whitegrid")

COLORWAY = ["#264653", "#2A9D8F", "#E9C46A", "#F4A261", "#E76F51", "#457B9D", "#D62828"]


In [ ]:
DATA_PATH = Path("/kaggle/input/student-performance-dataset/ultimate_student_productivity_dataset_5000.csv")
assert DATA_PATH.exists(), f"Dataset not found: {DATA_PATH.resolve()}"

df = pd.read_csv(DATA_PATH)
print(f"Loaded {DATA_PATH.name}")
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()


## 1) Segmentation Feature Engineering

We create a behavior-oriented feature space while keeping `exam_score` and `productivity_score` exclusively for post-cluster evaluation.


In [ ]:
seg_df = df.copy()

seg_df["leisure_hours"] = seg_df["social_media_hours"] + seg_df["gaming_hours"]
seg_df["learning_hours"] = seg_df["self_study_hours"] + seg_df["online_classes_hours"]
seg_df["digital_load_gap"] = seg_df["screen_time_hours"] - seg_df["learning_hours"]

feature_cols = [
    "study_hours",
    "self_study_hours",
    "online_classes_hours",
    "sleep_hours",
    "social_media_hours",
    "gaming_hours",
    "screen_time_hours",
    "exercise_minutes",
    "caffeine_intake_mg",
    "mental_health_score",
    "focus_index",
    "burnout_level",
    "part_time_job",
    "upcoming_deadline",
    "leisure_hours",
    "learning_hours",
    "digital_load_gap",
]

target_cols = ["productivity_score", "exam_score"]

display(seg_df[feature_cols + target_cols].describe().T)


In [ ]:
X = seg_df[feature_cols].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Clustering matrix shape: {X_scaled.shape}")


## 2) KMeans Model Selection

We combine three quality signals and require at least 4 clusters for actionable typology:

- Silhouette (higher better)
- Davies-Bouldin (lower better)
- Calinski-Harabasz (higher better)


In [ ]:
k_values = list(range(2, 10))
metric_rows = []

for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = km.fit_predict(X_scaled)
    metric_rows.append(
        {
            "k": k,
            "silhouette": silhouette_score(X_scaled, labels),
            "davies_bouldin": davies_bouldin_score(X_scaled, labels),
            "calinski_harabasz": calinski_harabasz_score(X_scaled, labels),
            "inertia": km.inertia_,
        }
    )

metrics_df = pd.DataFrame(metric_rows)

# Normalize metrics and build a blended quality score.
metrics_df["sil_norm"] = (
    (metrics_df["silhouette"] - metrics_df["silhouette"].min())
    / (metrics_df["silhouette"].max() - metrics_df["silhouette"].min())
)
metrics_df["dbi_inv"] = 1.0 / metrics_df["davies_bouldin"]
metrics_df["dbi_norm"] = (
    (metrics_df["dbi_inv"] - metrics_df["dbi_inv"].min())
    / (metrics_df["dbi_inv"].max() - metrics_df["dbi_inv"].min())
)
metrics_df["ch_norm"] = (
    (metrics_df["calinski_harabasz"] - metrics_df["calinski_harabasz"].min())
    / (metrics_df["calinski_harabasz"].max() - metrics_df["calinski_harabasz"].min())
)

metrics_df["quality_score"] = (
    0.40 * metrics_df["sil_norm"]
    + 0.30 * metrics_df["dbi_norm"]
    + 0.30 * metrics_df["ch_norm"]
)

display(metrics_df.round(4))


In [ ]:
fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=[
        "Silhouette (higher better)",
        "Inertia (lower better)",
        "Davies-Bouldin (lower better)",
        "Calinski-Harabasz (higher better)",
    ],
)

fig.add_trace(
    go.Scatter(x=metrics_df["k"], y=metrics_df["silhouette"], mode="lines+markers", line=dict(color=COLORWAY[1], width=3), name="Silhouette"),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(x=metrics_df["k"], y=metrics_df["inertia"], mode="lines+markers", line=dict(color=COLORWAY[0], width=3), name="Inertia"),
    row=1,
    col=2,
)
fig.add_trace(
    go.Scatter(x=metrics_df["k"], y=metrics_df["davies_bouldin"], mode="lines+markers", line=dict(color=COLORWAY[4], width=3), name="Davies-Bouldin"),
    row=2,
    col=1,
)
fig.add_trace(
    go.Scatter(x=metrics_df["k"], y=metrics_df["calinski_harabasz"], mode="lines+markers", line=dict(color=COLORWAY[5], width=3), name="Calinski-Harabasz"),
    row=2,
    col=2,
)

fig.update_layout(
    template="plotly_white",
    title="KMeans Selection Diagnostics",
    height=700,
    showlegend=False,
)
fig.update_xaxes(title_text="k")
fig.show()


In [ ]:
min_actionable_k = 4
candidates = metrics_df[metrics_df["k"] >= min_actionable_k].copy()

optimal_k = int(candidates.loc[candidates["quality_score"].idxmax(), "k"])
print(f"Chosen k (quality-score optimized, k >= {min_actionable_k}): {optimal_k}")

final_kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=30)
seg_df["cluster_id"] = final_kmeans.fit_predict(X_scaled)

cluster_sizes = seg_df["cluster_id"].value_counts().sort_index()
display(cluster_sizes.rename("students").to_frame())


## 3) Cluster Naming

We translate raw cluster IDs into interpretable student personas using standardized cluster signatures.


In [ ]:
naming_features = [
    "study_hours",
    "focus_index",
    "burnout_level",
    "leisure_hours",
    "sleep_hours",
    "mental_health_score",
    "part_time_job",
    "upcoming_deadline",
    "productivity_score",
    "exam_score",
]

cluster_profile = seg_df.groupby("cluster_id")[naming_features].mean()
global_mean = seg_df[naming_features].mean()
global_std = seg_df[naming_features].std().replace(0, np.nan)
z_profile = (cluster_profile - global_mean) / global_std


def propose_cluster_name(row):
    if row["burnout_level"] < -0.6 and row["exam_score"] > 0.3 and row["productivity_score"] > 0.2:
        return "High Achievers"
    if row["focus_index"] < -0.6 and row["leisure_hours"] > 0.6 and row["exam_score"] < -0.4:
        return "Struggling with Focus"
    if row["burnout_level"] > 0.5 and (row["upcoming_deadline"] > 0.2 or row["part_time_job"] > 0.2):
        return "Burnout Risk"
    if row["sleep_hours"] > 0 and row["burnout_level"] < 0 and row["exam_score"] > 0:
        return "Balanced Performers"
    return "Steady Performers"


proposed = z_profile.apply(propose_cluster_name, axis=1)

# Ensure names are unique if multiple clusters satisfy the same heuristic.
seen = {}
cluster_name_map = {}
for cid, label in proposed.items():
    seen[label] = seen.get(label, 0) + 1
    cluster_name_map[cid] = label if seen[label] == 1 else f"{label} {seen[label]}"

seg_df["cluster_name"] = seg_df["cluster_id"].map(cluster_name_map)

display(
    pd.DataFrame(
        {
            "cluster_id": list(cluster_name_map.keys()),
            "cluster_name": list(cluster_name_map.values()),
        }
    ).sort_values("cluster_id")
)


In [ ]:
cluster_summary = (
    seg_df.groupby(["cluster_id", "cluster_name"]).agg(
        n_students=("student_id", "count"),
        pct_students=("student_id", lambda s: 100 * len(s) / len(seg_df)),
        study_hours=("study_hours", "mean"),
        sleep_hours=("sleep_hours", "mean"),
        leisure_hours=("leisure_hours", "mean"),
        focus_index=("focus_index", "mean"),
        burnout_level=("burnout_level", "mean"),
        productivity_score=("productivity_score", "mean"),
        exam_score=("exam_score", "mean"),
    )
    .round(2)
    .sort_values("exam_score", ascending=False)
    .reset_index()
)

display(cluster_summary)


In [ ]:
fig = px.bar(
    cluster_summary.sort_values("n_students", ascending=False),
    x="cluster_name",
    y="n_students",
    color="cluster_name",
    text="pct_students",
    template="plotly_white",
    color_discrete_sequence=COLORWAY,
    title="Cluster Size and Share",
)
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.update_layout(showlegend=False, xaxis_title="Cluster", yaxis_title="Number of Students")
fig.show()


## 4) High-Quality Cluster Visual Diagnostics

- PCA scatter map for geometric separation.
- Signature heatmap for above/below-average trait reading.
- Radar chart for compact archetype storytelling.


In [ ]:
pca = PCA(n_components=2, random_state=42)
pca_2d = pca.fit_transform(X_scaled)

seg_df["pc1"] = pca_2d[:, 0]
seg_df["pc2"] = pca_2d[:, 1]

explained = pca.explained_variance_ratio_ * 100

fig = px.scatter(
    seg_df,
    x="pc1",
    y="pc2",
    color="cluster_name",
    size="focus_index",
    opacity=0.65,
    hover_data=["study_hours", "sleep_hours", "burnout_level", "productivity_score", "exam_score"],
    template="plotly_white",
    color_discrete_sequence=COLORWAY,
    title=f"PCA Cluster Map (PC1={explained[0]:.1f}% | PC2={explained[1]:.1f}%)",
)
fig.update_layout(xaxis_title="Principal Component 1", yaxis_title="Principal Component 2")
fig.show()


In [ ]:
heat_features = [
    "study_hours",
    "sleep_hours",
    "leisure_hours",
    "focus_index",
    "burnout_level",
    "mental_health_score",
    "productivity_score",
    "exam_score",
]

heat_raw = seg_df.groupby("cluster_name")[heat_features].mean()
heat_z = ((heat_raw - heat_raw.mean()) / heat_raw.std().replace(0, np.nan)).round(2)

fig = px.imshow(
    heat_z,
    text_auto=True,
    color_continuous_scale="RdBu_r",
    zmin=-2,
    zmax=2,
    aspect="auto",
    template="plotly_white",
    title="Cluster Signature Heatmap (Z-scores)",
)
fig.update_layout(xaxis_title="Feature", yaxis_title="Cluster")
fig.show()


In [ ]:
radar_features = ["study_hours", "sleep_hours", "leisure_hours", "focus_index", "burnout_level", "mental_health_score"]

radar_raw = seg_df.groupby("cluster_name")[radar_features].mean()
radar_min = radar_raw.min()
radar_max = radar_raw.max()
radar_norm = (radar_raw - radar_min) / (radar_max - radar_min).replace(0, np.nan)

fig = go.Figure()
for i, cname in enumerate(radar_norm.index):
    values = radar_norm.loc[cname].fillna(0).tolist()
    values = values + [values[0]]
    theta = radar_features + [radar_features[0]]

    fig.add_trace(
        go.Scatterpolar(
            r=values,
            theta=theta,
            fill="toself",
            name=cname,
            line=dict(color=COLORWAY[i % len(COLORWAY)], width=2),
            opacity=0.5,
        )
    )

fig.update_layout(
    template="plotly_white",
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    title="Normalized Behavioral Profile by Cluster",
)
fig.show()


## 5) Compare Clusters on Outcomes

Direct comparison of `productivity_score` and `exam_score` distributions by persona.


In [ ]:
outcomes_long = seg_df.melt(
    id_vars="cluster_name",
    value_vars=["productivity_score", "exam_score"],
    var_name="metric",
    value_name="score",
)

fig = px.box(
    outcomes_long,
    x="cluster_name",
    y="score",
    color="cluster_name",
    facet_col="metric",
    template="plotly_white",
    color_discrete_sequence=COLORWAY,
    title="Outcome Distributions by Cluster",
)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1].replace("_", " ").title()))
fig.update_layout(showlegend=False, height=520)
fig.show()


In [ ]:
outcome_rank = (
    seg_df.groupby("cluster_name")[["productivity_score", "exam_score"]]
    .mean()
    .round(2)
    .sort_values("exam_score", ascending=False)
    .reset_index()
)
display(outcome_rank)


## 6) Decision Tree: Explain Cluster Rules

The tree is a surrogate classifier trained to reproduce cluster labels from raw features.
High test accuracy indicates cluster logic can be summarized in compact if-then rules.


In [ ]:
X_tree = seg_df[feature_cols]
y_tree = seg_df["cluster_name"]

X_train, X_test, y_train, y_test = train_test_split(
    X_tree,
    y_tree,
    test_size=0.25,
    random_state=42,
    stratify=y_tree,
)

tree_model = DecisionTreeClassifier(max_depth=4, min_samples_leaf=100, random_state=42)
tree_model.fit(X_train, y_train)

train_acc = accuracy_score(y_train, tree_model.predict(X_train))
test_acc = accuracy_score(y_test, tree_model.predict(X_test))

print(f"Tree train accuracy: {train_acc:.3f}")
print(f"Tree test accuracy:  {test_acc:.3f}")


In [ ]:
y_pred = tree_model.predict(X_test)
labels = sorted(seg_df["cluster_name"].unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=labels, columns=labels)

fig = px.imshow(
    cm_df,
    text_auto=True,
    color_continuous_scale="Blues",
    template="plotly_white",
    title="Surrogate Tree vs Cluster Labels (Test Confusion Matrix)",
)
fig.update_layout(xaxis_title="Predicted Cluster", yaxis_title="True Cluster")
fig.show()


In [ ]:
importance = (
    pd.Series(tree_model.feature_importances_, index=feature_cols)
    .sort_values(ascending=False)
    .head(12)
    .reset_index()
)
importance.columns = ["feature", "importance"]

fig = px.bar(
    importance,
    x="importance",
    y="feature",
    orientation="h",
    template="plotly_white",
    color="importance",
    color_continuous_scale="Teal",
    title="Top Decision Tree Features Explaining Cluster Assignment",
)
fig.update_layout(yaxis=dict(categoryorder="total ascending"))
fig.show()


In [ ]:
plt.figure(figsize=(22, 10))
plot_tree(
    tree_model,
    feature_names=feature_cols,
    class_names=tree_model.classes_,
    filled=True,
    rounded=True,
    fontsize=9,
    max_depth=3,
)
plt.title("Decision Tree Rules (Depth <= 3 view)")
plt.show()


In [ ]:
print("Text-form rules:")
print(export_text(tree_model, feature_names=feature_cols))
